## Data Merging & Labeling

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Mount at /content/drive

In [ ]:
import pandas as pd

# Define Paths
drive_path = '/content/drive/MyDrive/fake_news_data/'

In [ ]:
# Define Helper Function
def combine_text(row):
    """Combine title and text into one string."""
    title = row.get('title', '')
    text = row.get('text', '')
    return f"{title} {text}".strip()

In [ ]:
# Load ISOT: True and Fake csv file
true_isot = pd.read_csv(drive_path + 'ISOT/True.csv')
fake_isot = pd.read_csv(drive_path + 'ISOT/Fake.csv')

# Add label column: 1 = fake, 0 = real
true_isot['label'] = 0
fake_isot['label'] = 1

# Add source
true_isot['source'] = 'ISOT'
fake_isot['source'] = 'ISOT'

# Combine title + text
true_isot['text'] = true_isot.apply(combine_text, axis=1)
fake_isot['text'] = fake_isot.apply(combine_text, axis=1)

# Keep only useful columns: title, text, subject, source, label
isot_clean = pd.concat([
    true_isot[['title', 'text', 'subject', 'source', 'label']],
    fake_isot[['title', 'text', 'subject', 'source', 'label']]
], ignore_index=True)

print("ISOT loaded. Shape:", isot_clean.shape)

In [ ]:
# Define label mapping for LIAR
liar_label_map = {
    'true': 0, 'mostly-true': 0, 'half-true': 0,
    'false': 1, 'pants-fire': 1, 'barely-true': 1
}

# Define column name (since the TSV file no header)
liar_cols = ['id','label','statement','subject','speaker','job_title',
             'state','party','barely_true','false','half_true',
             'mostly_true','pants_fire','context']

# Load each LIAR file
liar_train = pd.read_csv(drive_path + 'liar_dataset/train.tsv', sep='\t', header=None,
                         names=liar_cols)
liar_val = pd.read_csv(drive_path + 'liar_dataset/valid.tsv', sep='\t', header=None,
                       names=liar_train.columns)
liar_test = pd.read_csv(drive_path + 'liar_dataset/test.tsv', sep='\t', header=None,
                        names=liar_train.columns)

# Combine all LIAR statements
liar_all = pd.concat([liar_train, liar_val, liar_test], ignore_index=True)

# Keep only statement, subject and label
liar_all = liar_all[['statement', 'subject', 'label']].copy()
liar_all.rename(columns={'statement': 'text'}, inplace=True)

# Map labels to binary
liar_all['label'] = liar_all['label'].map(liar_label_map)

# Drop rows where label mapping failed (unknown labels)
liar_all = liar_all.dropna(subset=['label'])

# Add source and a placeholder for title
liar_all['source'] = 'LIAR'
liar_all['title'] = ''

# Reorder columns to match ISOT/Kaggle
liar_clean = liar_all[['title', 'text', 'subject', 'source', 'label']]

print("LIAR loaded. Shape:", liar_all.shape)

In [ ]:
# Load Kaggle dataset: True and Fake csv file
true_kaggle = pd.read_csv(drive_path + 'Kaggle/True.csv')
fake_kaggle = pd.read_csv(drive_path + 'Kaggle/Fake.csv')

true_kaggle['label'] = 0
fake_kaggle['label'] = 1

true_kaggle['source'] = 'Kaggle'
fake_kaggle['source'] = 'Kaggle'

# Combine title + text
true_kaggle['text'] = true_kaggle.apply(combine_text, axis=1)
fake_kaggle['text'] = fake_kaggle.apply(combine_text, axis=1)

kaggle_clean = pd.concat([
    true_kaggle[['title', 'text', 'subject', 'source', 'label']],
    fake_kaggle[['title', 'text', 'subject', 'source', 'label']]
], ignore_index=True)

print("Kaggle loaded. Shape:", kaggle_clean.shape)

In [ ]:
# Combine All Datasets
combined = pd.concat([isot_clean, liar_all, kaggle_clean], ignore_index=True)

# Remove duplicate texts (exact duplicates)
combined = combined.drop_duplicates(subset=['text'])

print("Combined dataset shape after dedup:", combined.shape)
print("Columns:", combined.columns.tolist())

In [ ]:
# Check Class Distribution
print("Label distribution:")
print(combined['label'].value_counts())

# Optional: plot distribution
import matplotlib.pyplot as plt
combined['label'].value_counts().plot(kind='bar')
plt.title('Fake vs Real News Distribution')
plt.xlabel('Label (0=Real, 1=Fake)')
plt.ylabel('Count')
plt.show()

In [ ]:
# Save Combined Dataset
output_path = drive_path + 'combined_fake_news.csv'
combined.to_csv(output_path, index=False)
print(f"Saved combined dataset to {output_path}")

## Exploratory Data Analysis (EDA)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
import re

# Load the dataset
df = pd.read_csv('/content/drive/MyDrive/fake_news_data/combined_fake_news.csv')

# Display basic info
print("Dataset shape:", df.shape)
print(df.head())

In [ ]:
# Visualize Class Distribution
# Count of real (0) vs fake (1)
class_counts = df['label'].value_counts().sort_index()
print(class_counts)

# Bar plot
plt.figure(figsize=(6,4))
sns.barplot(x=class_counts.index, y=class_counts.values, hue=class_counts.index,
            palette=['green', 'red'], legend=False)
plt.xticks([0,1], ['Real (0)', 'Fake (1)'])
plt.ylabel('Number of articles')
plt.title('Distribution of Real vs Fake News')
for i, count in enumerate(class_counts.values):
    plt.text(i, count + 200, str(count), ha='center')
plt.show()

# Pie chart
plt.figure(figsize=(6,6))
plt.pie(class_counts.values, labels=['Real', 'Fake'], autopct='%1.1f%%', colors=['green', 'red'])
plt.title('Percentage of Real vs Fake News')
plt.show()

In [ ]:
# Analyze Text Length Differences
# Add columns for text length (words and characters)
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))
df['char_count'] = df['text'].apply(lambda x: len(str(x)))

# Statistics per class
print("Word count statistics per class:")
print(df.groupby('label')['word_count'].describe())

# Boxplot for word counts
plt.figure(figsize=(8,5))
sns.boxplot(x='label', y='word_count', data=df, hue='label',
            palette=['green', 'red'], legend=False)
plt.xticks([0,1], ['Real', 'Fake'])
plt.title('Word Count Distribution by Label')
plt.ylabel('Number of words')
plt.show()

# Calculate 99th percentile of word_count
p99 = df['word_count'].quantile(0.99)
print(f"99th percentile of word count: {p99:.0f}")

# Histogram of word counts for both classes
plt.figure(figsize=(10,5))
sns.histplot(data=df, x='word_count', hue='label', bins=50, kde=True, palette=['green', 'red'])
plt.title('Histogram of Word Counts')
plt.xlim(0, p99)
plt.show()

In [ ]:
# Word clouds for fake vs real articles
# Download stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Define a function to clean text for word cloud
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words and len(w) > 2]
    return ' '.join(words)

# Separate fake and real texts
fake_text = ' '.join(df[df['label']==1]['text'].apply(clean_text))
real_text = ' '.join(df[df['label']==0]['text'].apply(clean_text))

# Generate word clouds
plt.figure(figsize=(16,8))

plt.subplot(1,2,1)
wordcloud_fake = WordCloud(width=800, height=400, background_color='black', colormap='Reds').generate(fake_text)
plt.imshow(wordcloud_fake, interpolation='bilinear')
plt.axis('off')
plt.title('Fake News - Most Common Words', fontsize=14)

plt.subplot(1,2,2)
wordcloud_real = WordCloud(width=800, height=400, background_color='black', colormap='Greens').generate(real_text)
plt.imshow(wordcloud_real, interpolation='bilinear')
plt.axis('off')
plt.title('Real News - Most Common Words', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze Subject Distribution (For Error Analysis)
# Count subjects per label
subject_counts = df.groupby(['subject', 'label']).size().unstack(fill_value=0)
# Show top 10 subjects overall
top_subjects = df['subject'].value_counts().head(10).index
subject_counts_top = subject_counts.loc[top_subjects]

# Plot
subject_counts_top.plot(kind='bar', figsize=(12,6), color=['green', 'red'])
plt.title('Number of Real vs Fake Articles by Subject (Top 10)')
plt.xlabel('Subject')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.legend(['Real', 'Fake'])
plt.show()

In [ ]:
# Save EDA Figures
plt.savefig('class_distribution.png', dpi=300, bbox_inches='tight')